# CancerNet: Breast Cancer Classification with TensorFlow

**Category:** Deep Learning → Neural Networks → Binary Classification  
**Source:** Reworked from `Abon_CPE312_HOA_9_2.ipynb` (the *9.1 supplementary* MLP exercise)  
**Goal:** Study an end-to-end, reproducible neural-network classification workflow with principled evaluation.

> **Educational demonstration only.** The Wisconsin Diagnostic Breast Cancer dataset is small and historical. These predictions are **not a diagnosis**, and this model is not validated for clinical deployment.

### Learning objectives
1. Examine numerical measurements and label encoding.
2. Separate training, validation, and test data before learning preprocessing statistics.
3. Create an input pipeline with `tf.data`, including batching and prefetching.
4. Compare a majority-class rule, a logistic regression baseline, and a multilayer perceptron (MLP).
5. Train with early stopping, examine learning curves, and investigate model capacity.
6. Select a decision threshold **on validation data only** and evaluate **once on the test set**.
7. Save, reload, and apply a trained model with its normalization layer attached.

**Data:** `sklearn.datasets.load_breast_cancer` — 569 observations, 30 numerical predictors, two labels. No separate CSV is needed. 

**Changes from the source notebook:** Better annotation, safer checks, separation of validation-based choices from held-out test evaluation, and explicitly optional ablation experiments. The original notebook is retained under `references/`.

## 0. Environment and reproducibility

Install requirements in a TensorFlow-enabled environment (for instance, Google Colab):

```bash
pip install -r requirements.txt
```

A random seed makes splitting and initialization more repeatable, though results can still vary across TensorFlow versions, CPU/GPU kernels, and hardware. No fixed test scores are asserted.

In [ ]:
# NumPy stores numeric arrays; pandas supports labelled tabular exploration.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

# scikit-learn supplies the *built-in* dataset and robust classification metrics.
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score,
    precision_recall_curve, roc_auc_score, roc_curve,
    average_precision_score,
)

# Fixing a seed helps reproduce the randomized train/test split,
# weight initialization and dataset shuffling used in this experiment.
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# A modest batch size works well for this small tabular dataset on a CPU.
BATCH_SIZE = 32
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")


## Part A. Understand the dataset and labels

The source dataset is the **Breast Cancer Wisconsin (Diagnostic)** dataset. Its 30 features summarize characteristics of cell nuclei derived from fine-needle aspirate imagery; these are **measurements**, not the images themselves.

**Important label convention:** scikit-learn encodes `0 = malignant`, `1 = benign`. For this tutorial we reverse that convention: `1 = malignant` becomes the **positive class**. This makes *recall* the fraction of actual malignant cases correctly flagged.

In [ ]:
# load_breast_cancer(as_frame=True) includes all 30 features and the target
# in a pandas DataFrame. We keep a DataFrame to inspect variable names.
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()

print("Shape (rows, features + target):", df.shape)
print("Target encoding from scikit-learn:", dict(enumerate(dataset.target_names)))
print("Missing cells:", int(df.isna().sum().sum()))
print("Number of unique classes:", df["target"].nunique())
display(df.head()) if "display" in globals() else print(df.head())


In [ ]:
# Different feature scales can cause some inputs to dominate optimization:
# a measurement with magnitude 1000 is numerically very different from 0.1.
# These statistics are descriptive only; we do not normalize using them.
feature_stats = (
    df.drop(columns="target")
    .describe().T[["mean", "std", "min", "max"]]
    .sort_values("max", ascending=False)
)
print(feature_stats.head(8))

# Class counts reveal imbalance, which makes accuracy alone insufficient.
class_counts = df["target"].map({0: "malignant", 1: "benign"}).value_counts()
print("Original class counts:\n", class_counts)
ax = class_counts.plot(kind="bar", rot=0, title="Class distribution")
ax.set_ylabel("Number of observations")
plt.tight_layout()
plt.show()


In [ ]:
# X contains the 30 measured features; y is the binary outcome.
# TensorFlow layers operate efficiently on float32 arrays.
X = df.drop(columns="target").to_numpy(dtype=np.float32)

# Reversing the source labels is NOT a model prediction:
# it simply defines malignant as the class of interest (positive = 1).
y = (1 - df["target"]).to_numpy(dtype=np.float32)

assert X.shape == (569, 30) and y.shape == (569,)
assert int(y.sum()) == 212, "Expected 212 malignant observations."
assert X.dtype == y.dtype == np.float32
print(f"Features: {X.shape}; malignant prevalence: {y.mean():.1%}")


## Part B. Split first, then fit preprocessing on training data

A **stratified 60/20/20 split** gives separate responsibilities to the data:

| Subset | Role | Permitted activities |
|---|---|---|
| Training | Learn weights and normalization statistics | `adapt()`, `fit()` |
| Validation | Choose architecture, early stopping epoch, and threshold | selection and diagnostics |
| Test | Final independent estimate of performance | report only **after** decisions are frozen |

Why two `train_test_split` calls? Holding out 20% for test leaves 80%; taking 25% of the remaining 80% gives 20% of the original data for validation. `stratify` approximately preserves the malignant percentage in each subset.

In [ ]:
# First reserve the 20% test set before any model-related fitting.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

# Of the original 80% development set, reserve a quarter for validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25, stratify=y_train_full, random_state=SEED,
)

assert (len(X_train), len(X_val), len(X_test)) == (341, 114, 114)
print(f"Train / validation / test sizes: {len(X_train)} / {len(X_val)} / {len(X_test)}")
print("Malignant proportions:",
      {"train": round(float(y_train.mean()), 3),
       "validation": round(float(y_val.mean()), 3),
       "test": round(float(y_test.mean()), 3)})


In [ ]:
def make_normalizer():
    """Create a fresh layer that learns feature-wise mean and variance on training data."""
    # axis=-1 computes one mean/variance per input feature (30 here).
    # IMPORTANT: .adapt(X_train), not .adapt(X), prevents information
    # from the validation/test distributions leaking into training.
    normalizer = tf.keras.layers.Normalization(axis=-1)
    normalizer.adapt(X_train)
    return normalizer

# Verify the transformation on training data, where the statistics were fit.
normalizer = make_normalizer()
standardized_train = normalizer(X_train).numpy()
print("Largest absolute feature mean after scaling:",
      np.abs(standardized_train.mean(axis=0)).max())
print("Largest feature std deviation from 1:",
      np.abs(standardized_train.std(axis=0) - 1).max())
assert np.allclose(standardized_train.mean(axis=0), 0, atol=1e-3)
assert np.allclose(standardized_train.std(axis=0), 1, atol=1e-2)


### `tf.data` input pipelines

`tf.data.Dataset` pairs each feature vector with its label. **Shuffle only training data** to reduce ordering effects during gradient updates. Validation and test pipelines should have stable order. `.batch(32)` improves training throughput; `.prefetch(tf.data.AUTOTUNE)` overlaps data preparation with model execution.

In [ ]:
# A buffer equal to training-set size allows effective full-set shuffling.
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=len(X_train), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Deterministic order keeps validation metrics consistent and easy to debug.
val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

sample_x, sample_y = next(iter(train_ds))
assert sample_x.shape == (BATCH_SIZE, 30)
assert sample_y.shape == (BATCH_SIZE,)
print("Batch features:", sample_x.shape, "labels:", sample_y.shape)


## Part C. Define metrics and train two baselines

**Accuracy** measures total correct predictions, but here a false negative means a malignant case was not flagged. **Recall (sensitivity)** is `TP/(TP+FN)`; **precision** is `TP/(TP+FP)`.

The *majority-class* baseline always predicts benign and exposes why accuracy can hide failure on the medically important class. A one-layer sigmoid model corresponds to **logistic regression**, giving a simple learned baseline.

ROC AUC evaluates probability ranking across thresholds, while **average precision** summarizes the precision-recall curve. A model's predicted probability is not automatically well calibrated; do not interpret it as a verified patient-level risk.

In [ ]:
def make_metrics():
    """Make independent stateful metric objects for each Keras model."""
    # Stateful metrics accumulate batch-level counts and should not be
    # reused across separately compiled models.
    return [
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ]

def summarize(name, y_true, y_prob, threshold=0.5):
    """Return comparable binary metrics with malignant treated as positive (1)."""
    y_true = np.asarray(y_true, dtype=int).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)
    if len(y_true) != len(y_prob):
        raise ValueError("Expected one probability per true label.")

    # A probability becomes a binary decision only after applying a threshold.
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": name,
        "threshold": round(float(threshold), 4),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "avg_precision": average_precision_score(y_true, y_prob),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

# A 0.0 probability for every observation means 'always benign'.
# The test set is NOT used at this point; final baseline results come later.
print("Reference malignant fraction in training:", float(y_train.mean()))


In [ ]:
# Logistic regression is equivalent to a linear weighted sum followed by
# a sigmoid. It cannot learn nonlinear feature interactions on its own.
tf.keras.utils.set_random_seed(SEED)
baseline = tf.keras.Sequential([
    tf.keras.Input(shape=(30,)),
    make_normalizer(),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="logistic_regression_baseline")

# Binary cross-entropy penalizes incorrect probability assignments;
# Adam optimizes the layer's 30 weights and 1 bias via gradients.
baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=make_metrics(),
)
baseline_history = baseline.fit(
    train_ds, validation_data=val_ds, epochs=100, verbose=0
)
print("Baseline trainable parameters:",
      int(np.sum([np.prod(w.shape) for w in baseline.trainable_weights])))
print("Final baseline validation loss:", baseline_history.history["val_loss"][-1])


## Part D. Design a multilayer perceptron (MLP)

Architecture: `30 inputs → Normalization → Dense(64, ReLU) → Dropout(0.3) → Dense(32, ReLU) → Dropout(0.3) → Dense(1, sigmoid)`.

Each **Dense** layer learns weighted combinations of input features. ReLU introduces nonlinearity so the network can represent patterns a logistic model cannot. **Dropout** randomly disables a portion of hidden activations during training, which can reduce overfitting. The final sigmoid produces a value in `(0, 1)` for the malignant class.

The parameters in this architecture were chosen as a practical example from the source notebook—not as clinically optimized settings.

In [ ]:
# Reset the seed before initializing weights to reduce random variation.
tf.keras.utils.set_random_seed(SEED)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(30,)),
    make_normalizer(),             # Saved inside model: accepts raw inputs.
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.30),  # Dropout is active during training only.
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="cancernet")

# Sanity-check architecture without assuming a version-specific parameter count.
dense_layers = [layer for layer in model.layers
                if isinstance(layer, tf.keras.layers.Dense)]
assert len(dense_layers) == 3
assert dense_layers[-1].units == 1
assert dense_layers[-1].activation.__name__ == "sigmoid"
model.summary()


### Model design card (filled for this implementation)

| Design element | Choice | Reason |
|---|---|---|
| Hidden layers | 64 and 32 neurons | Moderate capacity for 30 tabular measurements |
| Activation | ReLU | Learns nonlinear patterns and avoids saturation for positive activations |
| Regularization | 30% dropout and early stopping | Attempts to limit overfitting on a 341-row training set |
| Output and loss | One sigmoid unit + binary cross-entropy | Appropriate for malignant/benign classification |
| Optimizer | Adam, learning rate `0.001` | Adaptive gradient updates with a conservative step size |
| Highest-priority metric | Malignant-class recall, subject to validation precision ≥ 0.90 | Fewer missed malignant cases while controlling flagged-case burden |

**Clinical caveat:** A validation precision target is a classroom decision policy, not an established clinical safety threshold.

## Part E. Train with early stopping and inspect learning curves

Keras tracks a loss for each epoch. If training loss keeps improving while validation loss gets worse, that indicates **overfitting**. Early stopping watches validation loss and restores parameters from its best observed epoch. Training for "up to 300 epochs" does **not** mean the model must use all 300.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=make_metrics(),
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,           # Stop after 20 successive epochs without improvement.
    restore_best_weights=True,  # Keep the checkpoint with best val_loss.
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=300,
    callbacks=[early_stop],
    verbose=0,
)
assert "val_loss" in history.history and len(history.history["loss"]) <= 300
print("Epochs trained:", len(history.history["loss"]))
print("Best observed validation loss:", min(history.history["val_loss"]))


In [ ]:
def plot_history(training_history, metrics=("loss", "auc", "recall")):
    """Compare optimization on training data to generalization on validation data."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(15, 3.8))
    for axis, metric in zip(axes, metrics):
        # A train/validation gap can signal overfitting. Some stochastic
        # effects (dropout and small samples) can also affect this gap.
        axis.plot(training_history.history[metric], label="training")
        axis.plot(training_history.history[f"val_{metric}"], label="validation")
        axis.set(xlabel="Epoch", ylabel=metric, title=metric.title())
        axis.grid(alpha=0.25)
        axis.legend()
    plt.tight_layout()
    plt.show()

plot_history(history)


**Learning-curve interpretation guide:** Compare the train and validation losses near the epoch where `val_loss` is lowest. A sustained rise in validation loss alongside falling training loss can indicate overfitting; high losses on both may indicate underfitting. Because the validation set has only 114 cases, small fluctuations should not be overinterpreted. This notebook does not pre-fill an interpretation with unverified measurements.

## Part G. Optional ablation experiment — validation data only

Ablation compares controlled variants (depth, width, dropout). Each candidate is freshly initialized and trained on the **same training set**. Candidates are ranked by **validation loss** and are **not** judged using the test set.

The original source included five variants; these are retained, but the full ablation is opt-in because repeatedly fitting neural networks can be slow on a CPU. The primary model remains the fixed 64→32 architecture; if you select another architecture, choose it on validation data **before any final test evaluation**.

In [ ]:
def build_model(hidden_units=(64, 32), dropout_rate=0.30, name="mlp_variant"):
    """Create an independent architecture for validation-only comparison."""
    candidate = tf.keras.Sequential(name=name)
    candidate.add(tf.keras.Input(shape=(30,)))
    candidate.add(make_normalizer())  # Training-only preprocessing again.
    for units in hidden_units:
        candidate.add(tf.keras.layers.Dense(units, activation="relu"))
        candidate.add(tf.keras.layers.Dropout(dropout_rate))
    candidate.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    return candidate

# Change one factor at a time where possible for clearer comparisons.
variants = {
    "Base (64, 32), dropout 0.3": dict(hidden_units=(64, 32), dropout_rate=0.30),
    "No dropout": dict(hidden_units=(64, 32), dropout_rate=0.0),
    "Narrow (16,)": dict(hidden_units=(16,), dropout_rate=0.30),
    "Deep (128, 64, 32)": dict(hidden_units=(128, 64, 32), dropout_rate=0.30),
    "Wide (128,), dropout 0.5": dict(hidden_units=(128,), dropout_rate=0.50),
}
print("Variants configured:", len(variants))


In [ ]:
# Set True when you want to repeat the source notebook's ablation study.
RUN_ABLATION = False
ablation_results = pd.DataFrame()

if RUN_ABLATION:
    comparison_rows = []
    for variant_name, settings in variants.items():
        # Restart initialization, so changes reflect architecture rather
        # than a different random starting point (as much as possible).
        tf.keras.utils.set_random_seed(SEED)
        candidate = build_model(**settings)
        candidate.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
            metrics=make_metrics(),
        )
        candidate_history = candidate.fit(
            train_ds, validation_data=val_ds, epochs=300, verbose=0,
            callbacks=[tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=20, restore_best_weights=True
            )],
        )
        # The validation dataset is used here. The test set remains sealed.
        scores = candidate.evaluate(val_ds, return_dict=True, verbose=0)
        comparison_rows.append({
            "variant": variant_name, "params": candidate.count_params(),
            "epochs": len(candidate_history.history["loss"]),
            "val_loss": scores["loss"], "val_auc": scores["auc"],
            "val_recall_at_0.5": scores["recall"],
        })
    ablation_results = (
        pd.DataFrame(comparison_rows).set_index("variant").sort_values("val_loss")
    )
    print(ablation_results.round(4))
else:
    print("Ablation skipped. Set RUN_ABLATION=True to train all five variants.")


### Choosing the threshold *without touching test data*

The default threshold is `0.5`. The source exercise proposes a specific **classroom policy**: flag as many malignant observations as possible while attaining at least **90% validation precision**. Precision and recall are often in tension: lowering the threshold usually flags more observations and can increase recall, but it can also increase false alarms.

`precision_recall_curve(y_val, val_prob)` gives precision and recall at all observed candidate thresholds. We take the **lowest observed feasible threshold**, not the one with the best test score.

In [ ]:
# Select a threshold exclusively from validation labels and probabilities.
MIN_VALIDATION_PRECISION = 0.90
val_prob = model.predict(X_val, verbose=0).reshape(-1)
val_precision_curve, val_recall_curve, val_thresholds = precision_recall_curve(
    y_val, val_prob
)

# precision_recall_curve has one more precision value than thresholds.
# The final precision=1.0 entry has no corresponding cutoff.
feasible = val_thresholds[val_precision_curve[:-1] >= MIN_VALIDATION_PRECISION]
if feasible.size == 0:
    raise RuntimeError(
        "No observed validation cutoff achieved the precision requirement. "
        "Revisit the model or classroom decision policy without consulting test results."
    )
selected_threshold = float(feasible[0])

val_pred = (val_prob >= selected_threshold).astype(int)
val_precision = precision_score(y_val, val_pred, zero_division=0)
val_recall = recall_score(y_val, val_pred, zero_division=0)
assert val_precision + 1e-12 >= MIN_VALIDATION_PRECISION
print(f"Selected validation threshold: {selected_threshold:.4f}")
print(f"Validation malignant precision: {val_precision:.3f}")
print(f"Validation malignant recall: {val_recall:.3f}")


## Part F. Final holdout evaluation (only after model and threshold are set)

The following cells access the **test labels**. Run them only when your architecture, early-stopping rules, and validation-tuned threshold are fixed. Reporting multiple comparisons on the same held-out set is fine for description, but deciding to re-tune based on the test numbers invalidates its role as an independent final check.

The confusion matrix uses rows = actual and columns = predicted:

| | Predicted benign (0) | Predicted malignant (1) |
|---|---|---|
| Actual benign (0) | True negative | False positive |
| Actual malignant (1) | **False negative** | True positive |

Small test samples mean that apparent performance differences may be uncertain. Nothing here establishes clinical effectiveness.

In [ ]:
# Final evaluation: predict on the held-out test features after all choices.
baseline_prob = baseline.predict(X_test, verbose=0).reshape(-1)
model_prob = model.predict(X_test, verbose=0).reshape(-1)
majority_prob = np.zeros(len(y_test), dtype=float)  # Always predict benign.

# Same test examples and same underlying labels for every comparison.
final_rows = [
    summarize("Always benign", y_test, majority_prob),
    summarize("Logistic regression @0.5", y_test, baseline_prob),
    summarize("CancerNet @0.5", y_test, model_prob),
    summarize("CancerNet @validation threshold", y_test, model_prob,
              threshold=selected_threshold),
]
final_results = pd.DataFrame(final_rows).set_index("model")
print(final_results.round(4).to_string())


In [ ]:
# Confusion matrices expose *which* errors occur, especially missed
# malignant observations (false negatives). Accuracy cannot show this.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, cutoff, title in [
    (axes[0], 0.5, "CancerNet: default threshold"),
    (axes[1], selected_threshold, "CancerNet: validation-selected threshold"),
]:
    predicted = (model_prob >= cutoff).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test.astype(int), predicted,
        labels=[0, 1], display_labels=["benign", "malignant"],
        ax=axis, colorbar=False,
    )
    axis.set_title(title)
plt.tight_layout()
plt.show()

print("Malignant-class report (validation-selected threshold):")
print(classification_report(
    y_test.astype(int), (model_prob >= selected_threshold).astype(int),
    labels=[0, 1], target_names=["benign", "malignant"],
    zero_division=0, digits=3,
))


In [ ]:
# ROC curves summarize ranking quality at *all* thresholds, independent
# of one particular cutoff. In screening contexts, PR curves are also
# helpful because they show precision versus recall for positive cases.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for label, probabilities in [
    ("Logistic regression", baseline_prob),
    ("CancerNet", model_prob),
]:
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    axes[0].plot(fpr, tpr,
                 label=f"{label}: ROC AUC {roc_auc_score(y_test, probabilities):.3f}")
    p, r, _ = precision_recall_curve(y_test, probabilities)
    axes[1].plot(r, p, label=f"{label}: AP {average_precision_score(y_test, probabilities):.3f}")

axes[0].plot([0, 1], [0, 1], linestyle="--", label="Chance")
axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="Test ROC")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Test precision-recall")
for axis in axes:
    axis.legend(loc="best")
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Part H. Save the model and perform an example prediction

Saving a Keras model with its **Normalization layer** embedded makes deployment less error-prone: the caller supplies 30 raw features in the same order as training. The chosen **decision threshold is not automatically embedded in the model's sigmoid output**. Store that separately if deploying the decision rule.

This example uses existing dataset rows for an educational demonstration. It is **not suitable for use as a clinical application**.

In [ ]:
# The directory is created relative to the working directory. In Colab,
# outputs normally appear in the session filesystem unless you move them.
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "cancernet.keras"
model.save(str(MODEL_PATH))

# Loading rebuilds the architecture, weights and adapted normalization.
reloaded = tf.keras.models.load_model(str(MODEL_PATH))

# Confirm model serialization preserved predictions numerically.
saved_pred = model.predict(X_test[:10], verbose=0)
loaded_pred = reloaded.predict(X_test[:10], verbose=0)
assert np.allclose(saved_pred, loaded_pred, atol=1e-6)
print("Saved / reloaded model:", MODEL_PATH)

# Save the validation-selected classroom rule as a small text companion.
(MODEL_DIR / "validation_threshold.txt").write_text(
    f"{selected_threshold:.8f}\n", encoding="utf-8"
)


In [ ]:
def assess_case(raw_features, threshold=selected_threshold):
    """Return a class flag for a 30-feature measurement vector (study use only)."""
    raw_features = np.asarray(raw_features, dtype=np.float32).reshape(1, -1)
    if raw_features.shape[1] != 30:
        raise ValueError("Expected 30 ordered diagnostic measurements.")

    # Sigmoid output describes the model's score for the *malignant* label.
    # It should not be presented as a clinically calibrated risk estimate.
    score = float(reloaded.predict(raw_features, verbose=0)[0, 0])
    decision = "FLAG for human review" if score >= threshold else "not flagged"
    return score, decision

for index in range(3):
    score, decision = assess_case(X_test[index])
    true_label = "malignant" if y_test[index] == 1 else "benign"
    print(f"Case {index}: model score={score:.3f}; {decision}; actual={true_label}")


## Part I. Reflection and responsible use

Use **the numeric results from your own run** to answer these questions in your repository notes. Do not copy metric values from a different TensorFlow version or run without confirming them.

1. **Model complexity:** Did the MLP improve malignant-class recall or false-negative count over logistic regression? Was any improvement large enough to justify extra parameters and operational complexity?
2. **Accuracy trap:** Why can an always-benign model show a majority-class accuracy over 60% yet have 0 malignant recall?
3. **Threshold:** Did the validation-selected threshold decrease test false negatives? If it increased false positives, how did that change the trade-off?
4. **Data leakage:** Why must normalizer statistics, model choices, and threshold choices be derived without accessing test labels?
5. **Responsible application:** Identify safeguards such as external prospective validation across hospitals and demographics, drift monitoring, calibrated uncertainty review, human oversight, and regulatory evaluation.

**Interpretation reminder:** A successful class exercise is not evidence that the model is safe or effective for clinical decisions. The dataset represents a particular clinical collection, not every patient population.

## Bonus: custom Dense → BatchNorm → ReLU → Dropout block

The source notebook included an optional custom-layer challenge. Here, the layer explicitly passes Keras's `training` flag through BatchNormalization and Dropout. `@register_keras_serializable` plus `get_config()` makes the class easier to save and reload. Run this only if studying architecture customization.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="StudyNotes")
class DenseBNReLUBlock(tf.keras.layers.Layer):
    """Reusable dense block with batch normalization, ReLU and dropout."""
    def __init__(self, units, dropout_rate=0.30, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.dropout_rate = float(dropout_rate)
        self.dense = tf.keras.layers.Dense(self.units)
        self.batchnorm = tf.keras.layers.BatchNormalization()
        self.relu = tf.keras.layers.ReLU()
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)

    def call(self, inputs, training=None):
        # BatchNorm updates moving statistics during training but uses
        # stored statistics for inference; Dropout is off at inference.
        values = self.dense(inputs)
        values = self.batchnorm(values, training=training)
        values = self.relu(values)
        return self.dropout(values, training=training)

    def get_config(self):
        # get_config() allows Keras to reconstruct this class's settings.
        config = super().get_config()
        config.update({"units": self.units, "dropout_rate": self.dropout_rate})
        return config

RUN_CUSTOM_BLOCK = False
if RUN_CUSTOM_BLOCK:
    tf.keras.utils.set_random_seed(SEED)
    custom_model = tf.keras.Sequential([
        tf.keras.Input(shape=(30,)),
        make_normalizer(),
        DenseBNReLUBlock(64, 0.30),
        DenseBNReLUBlock(32, 0.30),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ], name="cancernet_custom_block")
    custom_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy", metrics=make_metrics()
    )
    custom_history = custom_model.fit(
        train_ds, validation_data=val_ds, epochs=300, verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=20, restore_best_weights=True
        )],
    )
    print("Custom-block validation metrics:",
          custom_model.evaluate(val_ds, return_dict=True, verbose=0))


### Further exercises

Try a **wide-and-deep Functional API** model; write a minimal `tf.GradientTape` training loop; or compare class weighting against validation-only threshold selection. For any such experiment, keep the test set unseen until you have finalized all hyperparameters. Record the validation metric, model size, and training conditions.

**Primary reference:** Original uploaded `Abon_CPE312_HOA_9_2.ipynb`. The uploaded PDF is titled *Hands-on Activity 9.1 TensorFlow Exercises*, covering tensor basics; it is background material, **not the instructions for the 9.2 notebook**.